<a href="https://colab.research.google.com/github/Shahul187/aml-alert-triage/blob/main/notebooks/03_features.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np

q = pd.read_parquet('/content/drive/MyDrive/aml-project/alert_queue.parquet')
q['Date'] = pd.to_datetime(q['Date'])

cutoff = pd.Timestamp('2023-06-01')
train = q[q['Date'] < cutoff].copy()
test  = q[q['Date'] >= cutoff].copy()

for name, d in [('TRAIN', train), ('TEST', test)]:
    print(f"{name}: {len(d):,} alerts | {d['Is_laundering'].sum():,} criminal "
          f"({d['Is_laundering'].mean()*100:.3f}%) | "
          f"{d['Date'].min().date()} to {d['Date'].max().date()}")

Mounted at /content/drive
TRAIN: 163,419 alerts | 3,355 criminal (2.053%) | 2022-10-07 to 2023-05-31
TEST: 56,544 alerts | 1,416 criminal (2.504%) | 2023-06-01 to 2023-08-23


In [2]:
hist = train.groupby('Sender_account').agg(
    hist_n=('Amount','size'),
    hist_mean=('Amount','mean'),
    hist_std=('Amount','std'),
    hist_max=('Amount','max'),
    hist_n_recv=('Receiver_account','nunique'),
).fillna(0)

hist['hist_recv_ratio'] = hist['hist_n_recv'] / hist['hist_n']

print(f"Profiles built for {len(hist):,} accounts")
print(hist.describe().round(1))

Profiles built for 44,792 accounts
        hist_n   hist_mean   hist_std    hist_max  hist_n_recv  \
count  44792.0     44792.0    44792.0     44792.0      44792.0   
mean       3.6     21167.8    12590.1     42160.0          1.9   
std        4.9    119045.9    86804.6    189103.9          2.2   
min        1.0        15.8        0.0        15.8          1.0   
25%        1.0      4274.9        0.0      4903.2          1.0   
50%        2.0      7340.1       39.9      8780.3          1.0   
75%        4.0     14580.1     2138.3     19726.2          2.0   
max       85.0  11837365.3  8915629.5  12618498.4         35.0   

       hist_recv_ratio  
count          44792.0  
mean               0.7  
std                0.3  
min                0.0  
25%                0.5  
50%                0.6  
75%                1.0  
max                1.0  


In [3]:
def add_features(d, hist):
    d = d.merge(hist, left_on='Sender_account', right_index=True, how='left')
    d['has_history'] = d['hist_n'].notna().astype(int)
    d[hist.columns] = d[hist.columns].fillna(0)

    d['amt_vs_mean'] = d['Amount'] / (d['hist_mean'] + 1)
    d['amt_vs_max']  = d['Amount'] / (d['hist_max'] + 1)
    d['amt_zscore']  = (d['Amount'] - d['hist_mean']) / (d['hist_std'] + 1)
    return d

train_f = add_features(train, hist)
test_f  = add_features(test, hist)

print(f"train_f: {train_f.shape} | test_f: {test_f.shape}")
print(f"Test rows with no known history: "
      f"{(test_f['has_history']==0).sum():,} "
      f"({(test_f['has_history']==0).mean()*100:.1f}%)")

train_f: (163419, 28) | test_f: (56544, 28)
Test rows with no known history: 17,947 (31.7%)


In [4]:
feats = ['amt_vs_mean','amt_vs_max','amt_zscore','hist_n',
         'hist_recv_ratio','has_history']

comp = train_f.groupby('Is_laundering')[feats].median().round(3).T
comp.columns = ['clean','criminal']
comp['ratio'] = (comp['criminal'] / (comp['clean'] + 0.001)).round(2)
print(comp)

                 clean  criminal  ratio
amt_vs_mean      0.996     1.000   1.00
amt_vs_max       0.665     0.581   0.87
amt_zscore      -0.198     0.000  -0.00
hist_n           6.000     6.000   1.00
hist_recv_ratio  0.500     0.714   1.43
has_history      1.000     1.000   1.00


In [5]:
recv_hist = train.groupby('Receiver_account').agg(
    r_n=('Amount','size'),
    r_mean=('Amount','mean'),
    r_n_senders=('Sender_account','nunique'),
).fillna(0)

recv_hist['r_sender_ratio'] = recv_hist['r_n_senders'] / recv_hist['r_n']

print(f"Receiver profiles: {len(recv_hist):,}")
print(recv_hist.describe().round(2))

Receiver profiles: 68,888
            r_n       r_mean  r_n_senders  r_sender_ratio
count  68888.00     68888.00     68888.00        68888.00
mean       2.37     28111.30         1.23            0.69
std        3.33    140385.83         1.46            0.31
min        1.00        10.49         1.00            0.04
25%        1.00       270.40         1.00            0.48
50%        2.00      5242.65         1.00            0.50
75%        3.00     11875.97         1.00            1.00
max       55.00  12618498.40        35.00            1.00


In [6]:
def add_recv(d, recv_hist):
    d = d.merge(recv_hist, left_on='Receiver_account', right_index=True, how='left')
    d['has_recv_history'] = d['r_n'].notna().astype(int)
    d[recv_hist.columns] = d[recv_hist.columns].fillna(0)
    return d

train_f = add_recv(train_f, recv_hist)
test_f  = add_recv(test_f,  recv_hist)

rfeats = ['r_n','r_n_senders','r_sender_ratio','r_mean','has_recv_history']
comp2 = train_f.groupby('Is_laundering')[rfeats].median().round(3).T
comp2.columns = ['clean','criminal']
comp2['ratio'] = (comp2['criminal'] / (comp2['clean'] + 0.001)).round(2)
print(comp2)

                     clean  criminal  ratio
r_n                  3.000     4.000   1.33
r_n_senders          1.000     1.000   1.00
r_sender_ratio       0.500     1.000   2.00
r_mean            6632.365  4648.843   0.70
has_recv_history     1.000     1.000   1.00


In [7]:
sender_set = set(train['Sender_account'])
recv_set   = set(train['Receiver_account'])
both = sender_set & recv_set

pair_counts = train.groupby(['Sender_account','Receiver_account']).size()

def add_final(d):
    d['recv_is_passthrough'] = d['Receiver_account'].isin(both).astype(int)
    d['sender_is_passthrough'] = d['Sender_account'].isin(both).astype(int)
    idx = pd.MultiIndex.from_arrays([d['Sender_account'], d['Receiver_account']])
    d['pair_seen'] = pair_counts.reindex(idx).fillna(0).values
    d['pair_is_new'] = (d['pair_seen'] == 0).astype(int)
    return d

train_f = add_final(train_f)
test_f  = add_final(test_f)

ffeats = ['recv_is_passthrough','sender_is_passthrough','pair_seen','pair_is_new']
comp3 = train_f.groupby('Is_laundering')[ffeats].mean().round(3).T
comp3.columns = ['clean','criminal']
comp3['ratio'] = (comp3['criminal']/(comp3['clean']+0.001)).round(2)
print(comp3)

                       clean  criminal  ratio
recv_is_passthrough    0.276     0.606   2.19
sender_is_passthrough  0.416     0.389   0.93
pair_seen              2.665     5.123   1.92
pair_is_new            0.000     0.000   0.00


In [8]:
FEATURES = ['Amount','alert_r1','alert_r2','alert_r3','alert_r4',
            'hist_n','hist_mean','hist_std','hist_max','hist_recv_ratio',
            'has_history','amt_vs_mean','amt_vs_max','amt_zscore',
            'r_n','r_mean','r_n_senders','r_sender_ratio','has_recv_history',
            'recv_is_passthrough','sender_is_passthrough','pair_seen','pair_is_new']

for d in (train_f, test_f):
    d['is_cash'] = d['Payment_type'].str.contains('Cash').astype(int)
    d['is_crossborder'] = (d['Sender_bank_location']!=d['Receiver_bank_location']).astype(int)
    d['same_currency'] = (d['Payment_currency']==d['Received_currency']).astype(int)

FEATURES += ['is_cash','is_crossborder','same_currency']

X_train, y_train = train_f[FEATURES], train_f['Is_laundering']
X_test,  y_test  = test_f[FEATURES],  test_f['Is_laundering']

print(f"X_train {X_train.shape} | X_test {X_test.shape}")
print(f"NaNs: {X_train.isna().sum().sum()}, {X_test.isna().sum().sum()}")

X_train (163419, 26) | X_test (56544, 26)
NaNs: 0, 0


In [9]:
FEATURES = [f for f in FEATURES if f != 'pair_is_new']

X_train, y_train = train_f[FEATURES], train_f['Is_laundering']
X_test,  y_test  = test_f[FEATURES],  test_f['Is_laundering']

base = '/content/drive/MyDrive/aml-project/'
train_f[FEATURES + ['Is_laundering','Laundering_type','Date']].to_parquet(base+'train_features.parquet', index=False)
test_f[FEATURES + ['Is_laundering','Laundering_type','Date']].to_parquet(base+'test_features.parquet', index=False)
pd.Series(FEATURES).to_csv(base+'feature_list.csv', index=False, header=['feature'])

print(f"{len(FEATURES)} features saved")
print(f"train {X_train.shape} | test {X_test.shape}")

25 features saved
train (163419, 25) | test (56544, 25)
